# Lab 6b — SOGS Pipeline: YouTube Video → GLOMAP → Self-Organizing Gaussians

**Course**: 3D Gaussian Splatting Compression  
**Instructor**: Nicholas McCarty (Upskilled Consulting)

This notebook runs the **Self-Organizing Gaussians (SOGS)** compression pipeline on the same
Bamburgh Castle YouTube footage used in Lab 6. The goal is a direct, apples-to-apples comparison:
same source video, same SfM reconstruction, different Gaussian trainer.

| # | Learning Objective | Section |
|---|---|---|
| LO1 | Understand what SOGS adds over vanilla 3DGS: SOM ordering, quantized SH, PLAS sorting | §5 Training |
| LO2 | Run the identical COLMAP/GLOMAP pipeline on the same source as Lab 6 | §3–4 SfM |
| LO3 | Measure SOGS training time vs. vanilla 3DGS reference | §6 Timing |
| LO4 | Quantify the compression benefit of SOM ordering via byte entropy analysis | §7 Compression |
| LO5 | Compare PSNR, file size, and gzip ratio between standard `.splat` and SOGS output | §8 Comparison |
| LO6 | Identify where SOGS fits in the research landscape and how to extend it | §9 Next Steps |

---

### Lab 6 Reference (Vanilla 3DGS, Bamburgh Castle, T4)

| Stage | Duration | Result |
|---|---|---|
| Video download + frame extraction | ~2 min | 1,031 frames @ 7 fps |
| COLMAP feature extraction + matching | ~12 min | ~96k verified pairs |
| GLOMAP reconstruction | ~17 min | 512/1,031 cameras |
| 3DGS training (30k iters) | ~14 min | PSNR 29.63 dB |
| `.splat` gzip compression ratio | — | ~1.5–2× |

> **SOGS advantage**: SOM ordering clusters spatially adjacent Gaussians together in the file.
> Adjacent Gaussians have similar attributes (position, colour, scale), so the byte sequence has
> lower entropy and gzip achieves ~3–5× compression vs. ~1.5–2× for opacity-volume sorted `.splat`.

## §0 — Install SOGS

SOGS (`fraunhoferhhi/Self-Organizing-Gaussians`) is a fork of 3DGS that replaces the standard
densification loop with a **Self-Organizing Map (SOM)** reordering step and adds **PLAS** (Parallel
Linear Assignment Sorting) for differentiable permutation learning.

Submodules installed here:
- **PLAS** — GPU-accelerated parallel linear assignment for SOM training
- **diff-gaussian-rasterization** — SOGS fork with SOM-aware gradient routing
- **simple-knn** — k-nearest-neighbour initialisation (shared with vanilla 3DGS)

We also clone vanilla `gaussian-splatting` inside the SOGS directory to resolve a `simple-knn`
version dependency — this is the same workaround used in the reference SOGS notebook.

In [ ]:
!git clone https://github.com/fraunhoferhhi/Self-Organizing-Gaussians.git --recursive
%cd Self-Organizing-Gaussians
!pip install -q ./submodules/PLAS
!pip install -q ./submodules/simple-knn
!pip install -q ./submodules/diff-gaussian-rasterization

In [ ]:
# Clone vanilla gaussian-splatting inside SOGS dir for simple-knn version resolution
!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive
%cd gaussian-splatting
!pip install -q ./submodules/simple-knn
%cd /content/

## §1 — Install COLMAP, GLOMAP & Python Dependencies

Identical setup to Lab 6: pre-built COLMAP/GLOMAP binaries from `nickmccarty/3dgs-workshop`,
runtime system libraries, and the Python packages required by both the pipeline utilities and SOGS.

In [ ]:
!pip install -q yt-dlp plyfile pandas
!apt-get install -y -q ffmpeg

!wget -q https://github.com/nickmccarty/3dgs-workshop/raw/refs/heads/master/colmap.zip
!wget -q https://github.com/nickmccarty/3dgs-workshop/raw/refs/heads/master/glomap.zip
!unzip -q colmap.zip
!unzip -q glomap.zip

!sudo apt-get install -y -q \
    libboost-program-options-dev libboost-filesystem-dev \
    libboost-graph-dev libboost-system-dev \
    libeigen3-dev libflann-dev libfreeimage-dev libmetis-dev \
    libgoogle-glog-dev libgtest-dev libsqlite3-dev libglew-dev \
    qtbase5-dev libqt5opengl5-dev libcgal-dev libceres-dev

!sudo apt-get install -y -q nvidia-cuda-toolkit nvidia-cuda-toolkit-gcc

In [ ]:
# SOGS-specific Python dependencies (versions validated against the reference SOGS notebook)
!pip install -q \
    "imagecodecs[all]==2023.9.18" \
    "hydra-core==1.3.2" \
    "kornia==0.7.3" \
    "opencv-python==4.10.0.84" \
    "scipy==1.14.1" \
    "screeninfo==0.8" \
    "tqdm==4.67.1"

In [ ]:
import os
os.environ['PATH'] += ':/content/colmap/build/src/colmap/exe'
os.environ['PATH'] += ':/content/glomap-1.0.0/build/glomap'
# Disable wandb — no account required to run this notebook
os.environ['WANDB_MODE'] = 'disabled'

!colmap -h 2>&1 | head -3

## §2 — Pipeline Timer

Same `stage()` context manager as Lab 6. Run `timing_summary()` after training to compare
elapsed times against the Lab 6 vanilla 3DGS reference.

In [ ]:
import time
import contextlib
import pandas as pd

_STAGES = []

@contextlib.contextmanager
def stage(name, expected_output=''):
    print(f"\n{'='*60}\n\u25b6  {name}\n{'='*60}")
    t0 = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - t0
        _STAGES.append({'Stage': name, 'Elapsed (s)': round(elapsed, 1), 'Key Output': expected_output})
        print(f'   \u2713 completed in {elapsed:.1f} s')

def timing_summary():
    df = pd.DataFrame(_STAGES)
    df['Elapsed (min)'] = (df['Elapsed (s)'] / 60).round(1)
    total = df['Elapsed (s)'].sum()
    print(df[['Stage', 'Elapsed (min)', 'Key Output']].to_string(index=False))
    print(f'\nTotal pipeline time: {total/60:.1f} min')
    return df

print('Timer ready.')

## §3 — Video Download & Frame Extraction

Same source video as Lab 6 (Bamburgh Castle, Nicholas McCarty). Extracting at **7 fps** yields
~1,031 frames — the exact same input the vanilla 3DGS run used, so any quality difference
between this notebook and Lab 6 is attributable to the trainer, not the input data.

```
data/
├── images/          ← ~1,031 frames @ 640×272
├── database.db      ← COLMAP feature database
└── sparse/0/        ← GLOMAP reconstruction
```

In [ ]:
from yt_dlp import YoutubeDL

URL = 'https://youtu.be/nefKdNDJR9M?si=4PDguQYUWTdTtZtG'

with stage('Video download', 'downloaded_video.mp4'):
    ydl_opts = {'format': 'best', 'outtmpl': './downloaded_video.%(ext)s'}
    with YoutubeDL(ydl_opts) as ydl:
        ydl.download([URL])

In [ ]:
with stage('Frame extraction — 7 fps', 'data/images/ — ~1,031 frames @ 640x272'):
    %cd data
    !mkdir -p images
    !ffmpeg -i ../downloaded_video.mp4 -qscale:v 1 -qmin 1 -vf "fps=7" images/image%04d.jpg

import glob
n_frames = len(glob.glob('images/*.jpg'))
print(f'Extracted {n_frames:,} frames')

## §4 — COLMAP Feature Extraction & Matching

Identical commands to Lab 6. PINHOLE camera model, single shared intrinsics, CUDA SIFT.
See Lab 6 §3 for the full discussion of flag choices.

In [ ]:
with stage('COLMAP feature extraction', 'database.db with SIFT keypoints'):
    !colmap feature_extractor \
       --database_path database.db \
       --image_path images \
       --ImageReader.camera_model PINHOLE \
       --ImageReader.single_camera 1 \
       --SiftExtraction.use_gpu 1

In [ ]:
with stage('COLMAP exhaustive matching', '~96k verified pairs in database.db'):
    !colmap exhaustive_matcher \
       --database_path database.db \
       --SiftMatching.use_gpu 1

## §5 — GLOMAP Global Reconstruction

GLOMAP global SfM writes a COLMAP-compatible binary reconstruction to `sparse/0/`.
SOGS reads this directly — same format as vanilla 3DGS.
See Lab 6 §4 for the full bundle adjustment derivation.

In [ ]:
with stage('GLOMAP global reconstruction', 'sparse/0/ — ~512 cameras, ~13k 3D points'):
    !glomap mapper \
        --database_path database.db \
        --image_path images \
        --output_path sparse

In [ ]:
%cd ..

# Quick registration check — full trajectory visualisation in Lab 6 §5
import struct, glob

def count_registered_cameras(images_bin):
    with open(images_bin, 'rb') as f:
        return struct.unpack('<Q', f.read(8))[0]

total_frames = len(glob.glob('data/images/*.jpg'))
registered   = count_registered_cameras('data/sparse/0/images.bin')
print(f'Registered {registered:,} / {total_frames:,} cameras  ({100*registered/total_frames:.1f}%)')

## §6 — SOGS Training

SOGS trains on the same COLMAP-format dataset as vanilla 3DGS (`dataset.source_path=./data`).
The config `ours_q_sh_local_test` enables:

| Option | What it does |
|---|---|
| `q_sh` | Quantize spherical harmonic coefficients to reduce per-Gaussian storage |
| `local` | Use local SOM neighbourhood updates (better spatial coherence than global) |
| `test` | Hold out every 8th registered camera for PSNR/SSIM/LPIPS evaluation |

The SOM reordering runs **alongside** 3DGS densification: every N iterations, Gaussians are
reindexed so that spatially neighbouring Gaussians are adjacent in memory. At export time,
the resulting `.ply` file has a byte sequence with much lower spatial entropy than the
opacity-volume sorted standard `.splat` — which is why gzip compresses it 3–5× better.

Output lands in `./sogs_output/` (Hydra `run.dir` override).

In [ ]:
with stage('SOGS training — ours_q_sh_local_test', 'sogs_output/ — compressed Gaussians'):
    !python ./Self-Organizing-Gaussians/train.py \
        --config-name ours_q_sh_local_test \
        dataset.source_path=./data \
        run.no_progress_bar=false \
        run.name=bamburgh-sogs \
        hydra.run.dir=./sogs_output

## §7 — Timing Summary

In [ ]:
timing_summary()

print()
print('Lab 6 vanilla 3DGS reference (T4):')
print('  Video download + extraction : ~2 min')
print('  COLMAP feat + match         : ~12 min')
print('  GLOMAP                      : ~17 min')
print('  3DGS training (30k iters)   : ~14 min')
print('  Total                       : ~45 min')

## §8 — Locate SOGS Output & Load PLY

In [ ]:
import glob as _glob
from plyfile import PlyData
import numpy as np

# SOGS writes point_cloud.ply under sogs_output/ in the same layout as vanilla 3DGS
ply_candidates = sorted(_glob.glob('sogs_output/**/point_cloud.ply', recursive=True))
if not ply_candidates:
    # Fallback: Hydra may have used a timestamped subdir
    ply_candidates = sorted(_glob.glob('outputs/**/point_cloud.ply', recursive=True))

assert ply_candidates, 'No SOGS output PLY found — did training complete?'
ply_path = ply_candidates[-1]
print(f'SOGS PLY: {ply_path}')

plydata = PlyData.read(ply_path)
vertex  = plydata['vertex']
fields  = [p.name for p in vertex.properties]
N       = len(vertex)

print(f'Gaussians  : {N:,}')
print(f'Fields ({len(fields)}): {fields[:10]} ...')

## §9 — Compression Analysis (LO4)

We measure byte entropy of the SOGS-ordered output and compare it against the vanilla 3DGS
opacity-volume sorted `.splat` from Lab 6.

**Why entropy matters**: gzip (LZ77 + Huffman) exploits byte-level correlations. Lower Shannon
entropy → shorter Huffman codes → better compression. SOM ordering creates spatial correlation
between consecutive Gaussians, which reduces entropy.

We convert the SOGS PLY to `.splat` using the same 32-byte encoder from Lab 6 so the comparison
is format-identical: both files are 32 bytes/Gaussian, differing only in byte ordering.

In [ ]:
import os, gzip, struct, math, matplotlib.pyplot as plt

SH_C0 = 0.28209479177387814

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def ply_to_splat_bytes(plydata):
    """Convert 3DGS-format PLY to 32-byte/Gaussian .splat binary.
    Input Gaussian order is preserved — do not sort here, so SOGS SOM order is kept.
    """
    v   = plydata['vertex']
    n   = len(v)
    pos = np.column_stack([v['x'], v['y'], v['z']]).astype(np.float32)
    sc  = np.exp(np.column_stack([v['scale_0'], v['scale_1'], v['scale_2']])).astype(np.float32)
    rgb = np.clip((0.5 + SH_C0 * np.column_stack([v['f_dc_0'], v['f_dc_1'], v['f_dc_2']])) * 255,
                  0, 255).astype(np.uint8)
    alpha = (sigmoid(np.array(v['opacity'])) * 255).astype(np.uint8)
    rgba  = np.column_stack([rgb, alpha])
    q     = np.column_stack([v['rot_0'], v['rot_1'], v['rot_2'], v['rot_3']]).astype(np.float32)
    q_n   = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-9)
    q_b   = np.clip((q_n * 128) + 128, 0, 255).astype(np.uint8)
    buf   = bytearray(n * 32)
    for i in range(n):
        o = i * 32
        struct.pack_into('<3f', buf, o,      *pos[i])
        struct.pack_into('<3f', buf, o + 12, *sc[i])
        buf[o + 24:o + 28] = rgba[i].tobytes()
        buf[o + 28:o + 32] = q_b[i].tobytes()
    return bytes(buf)

def byte_entropy(data):
    counts = [0] * 256
    for b in data:
        counts[b] += 1
    n = len(data)
    return -sum((c/n) * math.log2(c/n) for c in counts if c > 0)

# Convert SOGS PLY → .splat (preserving SOM order)
sogs_splat_bytes = ply_to_splat_bytes(plydata)
sogs_splat_gz    = gzip.compress(sogs_splat_bytes, compresslevel=9)

ply_size_mb        = os.path.getsize(ply_path) / 1e6
sogs_splat_mb      = len(sogs_splat_bytes) / 1e6
sogs_splat_gz_mb   = len(sogs_splat_gz) / 1e6
sogs_entropy       = byte_entropy(sogs_splat_bytes)

print(f'Gaussians              : {N:,}')
print(f'SOGS PLY (on disk)     : {ply_size_mb:.2f} MB')
print(f'SOGS .splat (SOM order): {sogs_splat_mb:.2f} MB   ({N} x 32 bytes)')
print(f'SOGS .splat gzipped    : {sogs_splat_gz_mb:.2f} MB')
print(f'Gzip ratio             : {sogs_splat_mb / sogs_splat_gz_mb:.2f}x')
print(f'Byte entropy           : {sogs_entropy:.3f} bits/byte  (max = 8.000)')
print()
print('Lab 6 reference (vanilla 3DGS, opacity-volume sort):')
print('  Byte entropy  : typically ~7.6–7.8 bits/byte')
print('  Gzip ratio    : typically ~1.5–2x')
print('  SOGS advantage: lower entropy from SOM ordering → better LZ77 match lengths')

In [ ]:
# Visualise byte value distribution — a flatter histogram = higher entropy = worse compression
byte_vals = np.frombuffer(sogs_splat_bytes, dtype=np.uint8)

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(byte_vals, bins=256, range=(0, 255), color='steelblue', edgecolor='none', alpha=0.8)
ax.set_xlabel('Byte value (0–255)')
ax.set_ylabel('Count')
ax.set_title(f'SOGS .splat byte distribution — entropy {sogs_entropy:.3f} bits/byte\n'
             f'Peaks indicate spatial correlation: adjacent Gaussians share attribute values')
plt.tight_layout()
plt.show()

## §10 — Side-by-Side Comparison with Vanilla 3DGS (LO5)

Fill in the Lab 6 results from your own run (or use the reference values from the header table).
The training PSNR from SOGS can be found in `sogs_output/cfg_args` or in the training log output above.

In [ ]:
import pandas as pd

# ── Fill in your Lab 6 results here ─────────────────────────────
vanilla_psnr_7k   = 27.75   # dB  (Lab 6 reference)
vanilla_psnr_30k  = 29.63   # dB  (Lab 6 reference)
vanilla_splat_mb  = None    # float: your Lab 6 .splat size in MB (or None to skip)
vanilla_gz_ratio  = None    # float: your Lab 6 gzip ratio (or None to skip)
# ────────────────────────────────────────────────────────────────

# Read SOGS final PSNR from training log (look for last 'PSNR' line in output above)
# If not captured, set manually:
sogs_psnr = None  # e.g. 28.4

rows = [
    ['Method',                  'Vanilla 3DGS (Lab 6)',     'SOGS (Lab 6b)'],
    ['Trainer',                 'gaussian-splatting',        'Self-Organizing-Gaussians'],
    ['PSNR @ 7k iters (dB)',    f'{vanilla_psnr_7k}',        '—'],
    ['PSNR @ 30k iters (dB)',   f'{vanilla_psnr_30k}',       str(sogs_psnr) if sogs_psnr else 'see log'],
    ['Gaussian ordering',       'Opacity-weighted volume',   'SOM spatial order'],
    ['SH encoding',             'Float32',                   'Quantized (q_sh)'],
    ['.splat size (MB)',        str(vanilla_splat_mb) or '—', f'{sogs_splat_mb:.2f}'],
    ['Gzip ratio',              str(vanilla_gz_ratio) or '—', f'{sogs_splat_mb / sogs_splat_gz_mb:.2f}x'],
    ['Byte entropy (bits/B)',   '~7.6–7.8 (typical)',        f'{sogs_entropy:.3f}'],
]

df = pd.DataFrame(rows[1:], columns=rows[0])
print(df.to_string(index=False))

## §11 — Save & View

Save the SOGS `.splat` file and view it in the same WebGL viewer used in Lab 6.

In [ ]:
sogs_splat_path = ply_path.replace('.ply', '_sogs.splat')
with open(sogs_splat_path, 'wb') as f:
    f.write(sogs_splat_bytes)
print(f'Saved: {sogs_splat_path}')
print(f'Size : {os.path.getsize(sogs_splat_path) / 1e6:.2f} MB')
print()
print('Download and drag into: https://antimatter15.com/splat/')
print('Compare to Lab 6 result at: https://nickmccarty.me/bamburgh-castle-splat')

In [ ]:
from google.colab import files
files.download(sogs_splat_path)

## §12 — Next Steps & Research Extensions (LO6)

This notebook and Lab 6 together form a minimal research scaffold. Below are concrete directions
for extending this work.

### 1. Benchmark on a Canonical Dataset

The YouTube video is a useful personal test but has no published ground-truth PSNR to compare against.
For publishable results, use a standard benchmark:

| Dataset | URL | Scenes | Notes |
|---|---|---|---|
| Tanks & Temples (truck, train) | `https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip` | 2 | Used in SOGS paper |
| MipNeRF-360 (outdoor/indoor) | `https://jonbarron.info/mipnerf360/` | 9 | High-quality diverse scenes |
| IMW 2020 Phototourism | `https://www.cs.ubc.ca/~kmyi/imw2020/data.html` | 25+ | Large-scale, crowd-sourced |
| Deep Blending | `https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/` | 2 | Indoor, challenging lighting |

For Tanks & Temples, replace `dataset.source_path=./data` with `dataset.source_path=./tandt/truck`
and the SOGS paper table gives you the published PSNR to compare against.

### 2. Tune the SOM Schedule

The `ours_q_sh_local_test` config uses default SOM update intervals. Key Hydra overrides to experiment with:

```bash
# Increase SOM update frequency (better ordering, slower training)
run.som_update_interval=50   # default ~100

# Change SH quantization bit-depth
run.sh_quantize_bits=8       # default 8; try 6 or 4 for more aggressive compression

# Disable quantization to isolate the SOM ordering benefit
--config-name ours_local_test
```

### 3. Automate the Comparison Loop

The two-notebook setup is the basis for a systematic ablation. With the scene reconstruction
shared (GLOMAP output), you can re-run training with different configs in the same Colab session:

```python
configs = ['ours', 'ours_q_sh', 'ours_q_sh_local_test']
for cfg in configs:
    !python ./Self-Organizing-Gaussians/train.py \
        --config-name {cfg} \
        dataset.source_path=./data \
        run.name={cfg} \
        hydra.run.dir=./outputs/{cfg}
```

Then collect the final PSNR from each output log and produce a single comparison table.

### 4. Depth Priors + SOGS

Lab 6 §Extended shows how Depth-Anything-V2 priors can improve quality on thin structures.
SOGS does not currently accept `--depth_path` natively, but depth maps can be integrated
as a custom loss term in the SOGS training loop — a viable research contribution.

### 5. Viewer Integration

Both Lab 6 and this notebook output standard `.splat` files viewable at `https://antimatter15.com/splat/`.
For an embedded viewer in a web page, see `splat.js` (antimatter15) or `gaussian-splatting-babylon`
(babylonjs). The SOGS output is a drop-in replacement — no viewer changes needed.

---
## Lab 6b Completion Checklist

- [ ] **LO1** — SOGS architecture understood: SOM ordering, quantized SH, PLAS sorting explained
- [ ] **LO2** — Pipeline ran end-to-end on the same YouTube video as Lab 6
- [ ] **LO3** — SOGS training time recorded; compared against Lab 6 vanilla 3DGS timing
- [ ] **LO4** — Byte entropy measured; distribution plot shows correlation peaks vs. Lab 6 flat distribution
- [ ] **LO5** — Comparison table filled in with actual PSNR and compression ratios from both notebooks
- [ ] **LO6** — At least one extension from §12 identified and written up as a one-paragraph research proposal

**Deliverable**: Drop both your Lab 6 and Lab 6b `.splat` files into `https://antimatter15.com/splat/`
and document one visual difference between the two renderings.